# 01 — Data & Embeddings

**Does one thing, nothing more:**
1. Downloads MovieLens 1M
2. Converts each movie to a text description
3. Encodes with Sentence-BERT
4. Caches the result

**output:** `data/movies.pkl` and `data/embeddings.npy`

In [ ]:
import numpy as np
import pandas as pd
import requests, zipfile, io
from pathlib import Path
from sentence_transformers import SentenceTransformer

DATA_DIR   = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

SBERT_MODEL = 'all-MiniLM-L6-v2'

print('imports OK')

### Step 1 — Download MovieLens 1M

In [ ]:
def download_movielens(data_dir: Path):
    if (data_dir / 'movies.dat').exists():
        print('Already downloaded.')
        return
    url = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'
    print('Downloading ...')
    r = requests.get(url)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    for name in z.namelist():
        if name.endswith('.dat'):
            fname = Path(name).name
            with z.open(name) as src, open(data_dir / fname, 'wb') as dst:
                dst.write(src.read())
    print('Done.')

download_movielens(DATA_DIR)

### Step 2 — Load & Build Item Text

In [ ]:
movies = pd.read_csv(
    DATA_DIR / 'movies.dat',
    sep='::',
    engine='python',
    names=['movie_id', 'title', 'genres'],
    encoding='latin-1'
)

movies['year']        = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
movies['title_clean'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()
movies['genres_clean']= movies['genres'].str.replace('|', ' ', regex=False)

# text to embed: title + genres
movies['text'] = movies['title_clean'] + ' | ' + movies['genres_clean']

print(f'Movies: {len(movies)}')
movies[['title_clean', 'genres_clean', 'text']].head(5)

### Step 3 — Embed

In [ ]:
EMB_PATH = DATA_DIR / 'embeddings.npy'

if EMB_PATH.exists():
    print('Loading cached embeddings ...')
    embeddings = np.load(EMB_PATH)
else:
    print(f'Encoding with {SBERT_MODEL} ...')
    model = SentenceTransformer(SBERT_MODEL)
    embeddings = model.encode(
        movies['text'].tolist(),
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True   # required for cosine similarity
    )
    np.save(EMB_PATH, embeddings)
    print('Saved to cache.')

print(f'Shape: {embeddings.shape}')  # expected shape: (3883, 384)

### Step 4 — Save movies dataframe

In [ ]:
movies.to_pickle(DATA_DIR / 'movies.pkl')
print('Saved: data/movies.pkl')
print('Saved: data/embeddings.npy')
print()
